In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
#read all words
words=open('../names.txt','r').read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [3]:
len(words)

32033

In [4]:
#get all the characters in the words, get char to 
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}

In [5]:
block_size=3
X,Y=[],[]
for w in words:
    #print(w)
    context=[0]*block_size
    for ch in w + '.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context),'--------->',itos[ix])
        context=context[1:]+[ix] #crop and append
X=torch.tensor(X)
Y=torch.tensor(Y)

In [6]:
C=torch.randn((27,2))

In [7]:
X.shape,Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [31]:
g=torch.Generator().manual_seed(2147483647)
C=torch.randn((27,2),generator=g)
W1=torch.randn((6,100),generator=g)
b1=torch.randn(100,generator=g)
W2=torch.randn((100,27),generator=g)
b2=torch.randn(27,generator=g)
parameters=[C,W1,b1,W2,b2]

In [32]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [33]:
for p in parameters:
    p.requires_grad=True

In [38]:
for i in range(100):
    #minibatch construct
    ix=torch.randint(0,X.shape[0],(32,))
    #forward pass
    emb=C[X[ix]]
    h=torch.tanh(emb.view(-1,6)@ W1 + b1)
    logits=h@W2 +b2
    # counts=logits.exp()
    # prob=counts/counts.sum(1,keepdims=True)
    # loss=-prob[torch.arange(32),Y)].log().mean()
    loss=F.cross_entropy(logits,Y[ix])
        #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()
    #update
    for p in parameters:
        p.data+= -0.1*p.grad

In [39]:
emb=C[X]
h=torch.tanh(emb.view(-1,6)@W1+b1)
logits= h @ W2 + b2
loss=F.cross_entropy(logits,Y)
loss

tensor(2.8757, grad_fn=<NllLossBackward0>)